# vLLM serving benchmark on a Kaggle T4

Question: how do time to first token (TTFT), time per output token (TPOT) and throughput change as concurrent requests grow, and does vLLM's KV cache size match the sizing math?

Setup: one Tesla T4 (16 GB, Turing, no bf16, so fp16), `Qwen/Qwen2.5-1.5B-Instruct`, vLLM OpenAI-compatible server, `vllm bench serve` with random 512-token prompts and 128 output tokens.

In [ ]:
# 1. Install vLLM (into the Kaggle VM, not your laptop)
!pip install -q vllm 2>&1 | tail -3
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__)"
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# 2. Sizing math before running anything
import json, subprocess, urllib.request
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_LEN, GPU_UTIL, BYTES = 4096, 0.90, 2  # fp16 = 2 bytes
cfg = json.load(urllib.request.urlopen(f"https://huggingface.co/{MODEL}/resolve/main/config.json"))
params = json.load(urllib.request.urlopen(f"https://huggingface.co/api/models/{MODEL}"))["safetensors"]["total"]
L, H, KVH = cfg["num_hidden_layers"], cfg["num_attention_heads"], cfg["num_key_value_heads"]
D = cfg.get("head_dim") or cfg["hidden_size"] // H
kv_per_token = 2 * L * KVH * D * BYTES  # K and V, every layer, every KV head
total_gib = int(subprocess.check_output(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"]).split()[0]) / 1024
weights_gib = params * BYTES / 2**30
OVERHEAD_GIB = 1.0  # guess: activations + CUDA graphs; the real number comes from vLLM's log below
pred_tokens = (total_gib * GPU_UTIL - weights_gib - OVERHEAD_GIB) * 2**30 / kv_per_token
print(f"params {params/1e9:.2f} B -> weights {weights_gib:.2f} GiB in fp16")
print(f"layers {L}, query heads {H}, KV heads {KVH} (GQA {H//KVH}:1), head_dim {D}")
print(f"KV cache per token = 2 x {L} x {KVH} x {D} x {BYTES} B = {kv_per_token:,} B ({kv_per_token/1024:.0f} KiB)")
print(f"one {MAX_LEN}-token request needs {kv_per_token*MAX_LEN/2**20:.0f} MiB of KV cache")
print(f"GPU {total_gib:.1f} GiB x {GPU_UTIL} - weights - {OVERHEAD_GIB} GiB guess -> about {pred_tokens:,.0f} KV tokens predicted")
print(f"-> about {pred_tokens/MAX_LEN:.0f} full-length requests fit at once")

In [ ]:
# 3. Start the vLLM OpenAI-compatible server and wait for /health
import time
log_path = "/kaggle/working/vllm_server.log"
srv = subprocess.Popen(["vllm", "serve", MODEL, "--dtype", "half", "--max-model-len", str(MAX_LEN),
                        "--gpu-memory-utilization", str(GPU_UTIL), "--port", "8000"],
                       stdout=open(log_path, "w"), stderr=subprocess.STDOUT)
t0 = time.time()
while True:
    try:
        urllib.request.urlopen("http://localhost:8000/health"); break
    except Exception:
        if srv.poll() is not None:
            print(open(log_path).read()[-4000:]); raise SystemExit("server exited")
        if time.time() - t0 > 1200: raise SystemExit("server not ready after 20 min")
        time.sleep(2)
print(f"server ready after {time.time()-t0:.0f} s")
!grep -iE "backend|weights|model loading|kv cache|maximum concurrency|cuda graph" /kaggle/working/vllm_server.log | cut -c1-220

In [ ]:
# 4. Concurrency sweep with vLLM's own benchmark client
rows = []
for c in [1, 4, 16, 64]:
    name = f"bench_c{c}.json"
    r = subprocess.run(["vllm", "bench", "serve", "--backend", "vllm", "--base-url", "http://localhost:8000",
                        "--model", MODEL, "--dataset-name", "random", "--random-input-len", "512",
                        "--random-output-len", "128", "--ignore-eos", "--num-prompts", str(max(16, 4 * c)),
                        "--max-concurrency", str(c), "--seed", "0", "--save-result",
                        "--result-dir", "/kaggle/working", "--result-filename", name],
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:]); break
    d = json.load(open(f"/kaggle/working/{name}"))
    rows.append({"concurrency": c, "requests": d["completed"],
                 "TTFT median ms": round(d["median_ttft_ms"]), "TTFT p99 ms": round(d["p99_ttft_ms"]),
                 "TPOT median ms": round(d["median_tpot_ms"], 1),
                 "output tok/s": round(d["output_throughput"]), "requests/s": round(d["request_throughput"], 2)})
    print(rows[-1])
import pandas as pd
df = pd.DataFrame(rows); df.to_csv("/kaggle/working/vllm_t4_sweep.csv", index=False)
print(df.to_markdown(index=False))
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
# 5. Stop the server so the GPU quota stops counting work
srv.terminate(); srv.wait(timeout=60); print("server stopped")

## What to look for

- **TTFT** rises with concurrency: new prompts wait for prefill behind others (queueing plus compute-bound prefill).
- **TPOT** rises slowly: decode is memory-bandwidth bound, and continuous batching shares each weight read across more sequences.
- **Output tok/s** rises with concurrency until compute or KV cache space runs out. That is the latency vs throughput trade-off.
- Compare the predicted KV tokens in step 2 with vLLM's "GPU KV cache size" line in step 3.